<a href="https://colab.research.google.com/github/giulia-belgiovine/Memorability/blob/main/Menorability_ContinualLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image
from csv import reader
from os.path import expanduser
import os, sys, copy, glob, tqdm, argparse
import time
import csv
import pickle

In [ ]:
# Torch
import torch
from torch import nn
from torch.optim import Adam, lr_scheduler
from torch.nn import CrossEntropyLoss
from torchvision import datasets, models, transforms
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.utils.data.dataset import random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.transforms import ToTensor, RandomResizedCrop, RandomHorizontalFlip,  RandomRotation, Compose, Normalize, RandomCrop, Resize, CenterCrop

# torch.use_deterministic_algorithms(True)


## Load Dataset

In [ ]:
np.random.seed(0)

In [ ]:
csv_path = "/content/drive/MyDrive/BMM_2023/projects/Datasets/MemCat_data"
imgs_path = "/content/drive/MyDrive/BMM_2023/projects/Datasets/MemCat"

class_dict = {"animal": 0, "food":1, "landscape":2, "sports":3, "vehicle":4}

memcat_frame = pd.read_csv(os.path.join(csv_path,"memcat_image_data.csv"))
memcat_frame[:3]

,Unnamed: 0,image_file,category,subcategory,current_height,current_width,source,searched_label,resize_factor,H,FA,n_resp,memorability_wo_fa_correction,memorability_w_fa_correction
0,1,000000003481.jpg,animal,bear,316.0,500.0,coco,bear,1.0,55,4,98,0.561224,0.520408
1,2,000000005745.jpg,animal,bear,427.0,640.0,coco,bear,1.0,64,4,81,0.790123,0.740741
2,3,000000011552.jpg,animal,bear,427.0,640.0,coco,bear,1.0,75,5,104,0.721154,0.673077


In [ ]:
n_samples_val = 200            #Take 200 samples per ach class in the test set
samples_per_class = 800        #Once done the median split on the remaining train_set, take 800 samples for each score class ('high'/'low')
final_list_test = []
final_list_train = []
np.random.seed(0)

for k, v in class_dict.items():

    subframe = memcat_frame.loc[memcat_frame["category"]==k]

    #Take 200 random samples from test
    df_val = subframe.sample(n=n_samples_val)
    df_train = subframe.drop(df_val.index)
    # print("Lenght of test dataframe: {}. Lenght of the train dataframe: {}".format(len(df_val), len(df_train)))

    # Perform a median split based on mem. score on the remaining train set.
    # Top 50th percentile put in group '2'. Bottom 50th percentile be put in group '1'
    score = df_train["memorability_w_fa_correction"]
    df_train["median_split"] = (score < score.quantile()).replace({True:1, False:2})

    low_score_df = df_train.loc[df_train["median_split"] == 1]
    high_score_df = df_train.loc[df_train["median_split"] == 2]
    # print("Lenght of first median split: {}. Lenght of second median split: {}.".format(len(low_score_df), len(high_score_df)))

    # Take 800 from low_score and 800 from hig score to remove the ones too close to median
    composed_dataset = pd.concat([high_score_df[:samples_per_class], low_score_df[-samples_per_class:]])
    composed_dataset = composed_dataset.sort_values(by='memorability_w_fa_correction', ascending=True)

    final_list_train.append(composed_dataset)
    final_list_test.append(df_val)

train_df = pd.concat(final_list_train) #len of 1600*5=8000
test_df = pd.concat(final_list_test)   #len of 200*5=1000
print("Total lenght of: Train Dataset --> {}. Test Dataset --> {}".format(len(train_df), len(test_df)))

Total lenght of: Train Dataset --> 8000. Test Dataset --> 1000


In [ ]:
#check number of samples per categories
for k, v in class_dict.items():

  n_val = len(test_df.loc[test_df['category'] == k])
  n_train = len(train_df.loc[train_df['category'] == k])

  print('Number of samples for category {}: Train --> {}. Test --> {}'.format(k, n_train, n_val))

Number of samples for category animal: Train --> 1600. Test --> 200
Number of samples for category food: Train --> 1600. Test --> 200
Number of samples for category landscape: Train --> 1600. Test --> 200
Number of samples for category sports: Train --> 1600. Test --> 200
Number of samples for category vehicle: Train --> 1600. Test --> 200


In [ ]:
#Just making a more compact train and test Dataframe
def save_compact_df(dataframe):
  dictlist = []
  for i, row in dataframe.iterrows():

      imgstr = row["image_file"]
      #index = i
      img_path = os.path.join(imgs_path, row['category'], row['subcategory'], row['image_file'])
      subcategory = row['subcategory']
      label = row['category']
      score = row["memorability_w_fa_correction"]

      # add these to a dictionary:
      row = {'image_name': imgstr, 'label': label, 'subcategory':subcategory, 'score': score, 'img_path': img_path}  #'image_index': index,

      # append dictionary to running list of rows
      dictlist.append(row)

  df_compact = pd.DataFrame(dictlist)
  return df_compact

train_compact = save_compact_df(train_df)  #the dataset is sorted per category
test_compact = save_compact_df(test_df)

In [ ]:
# CREATE DATASET FOR DIFFERENT EXPERIMENTS
np.random.seed(0)
df_exp1_list = []
df_exp2_list = []
df_exp3_list = []

for k in class_dict.keys():

  df = train_compact.loc[train_compact['label'] == k]

  #Experiment 1
  df_rnd = df.sample(n=samples_per_class)  #created by random sampling of 800 imgs per classes
  df_exp1_list.append(df_rnd)

  #Experiment 2
  df_high = df[-samples_per_class:]       #created by sampling highest score 800 imgs per classes
  df_exp2_list.append(df_high)

  #Experiment 3
  df_low = df[:samples_per_class]         #created by sampling lowest score 800 imgs per classes
  df_exp3_list.append(df_low)

  print("Low-score range for class {} --> [{} : {}]".format(k, df_low['score'].min(), df_low['score'].max()))
  print("High-score range fpr class {} --> [{} : {}]".format(k, df_high['score'].min(), df_high['score'].max()))


## DATASET PER EXPERIMENTS
train_exp_rnd = pd.concat(df_exp1_list).sample(frac = 1)
train_exp_high = pd.concat(df_exp2_list).sample(frac = 1)
train_exp_low = pd.concat(df_exp3_list).sample(frac = 1)

Low-score range for class animal --> [0.333333333333333 : 0.732558139534884]
High-score range fpr class animal --> [0.732673267326733 : 0.97752808988764]
Low-score range for class food --> [0.423076923076923 : 0.813559322033898]
High-score range fpr class food --> [0.813725490196078 : 0.975]
Low-score range for class landscape --> [0.144067796610169 : 0.521739130434783]
High-score range fpr class landscape --> [0.522222222222222 : 0.910714285714286]
Low-score range for class sports --> [0.323529411764706 : 0.715686274509804]
High-score range fpr class sports --> [0.715789473684211 : 0.959183673469388]
Low-score range for class vehicle --> [0.280898876404494 : 0.700934579439252]
High-score range fpr class vehicle --> [0.701030927835051 : 0.941860465116279]


In [ ]:
# SELECT THE DATASET FOR THE EXPERIMENT
# np.random.seed(0)

# train_exp = {'name': 'Random', 'df': train_exp_rnd.reset_index()}
# train_exp = {'name': 'High_Score', 'df': train_exp_high.reset_index()}
# train_exp = {'name': 'Low_Score', 'df': train_exp_low.reset_index()}

test_exp = test_compact.sample(frac = 1).reset_index()

## Continual Leaning Experiment

In [ ]:
# Use Avalanche Library
!pip install avalanche-lib

In [ ]:
# Avalanche
from torch.optim import SGD
from avalanche.benchmarks import nc_benchmark
from avalanche.benchmarks.utils import make_classification_dataset
from avalanche.benchmarks.classic import SplitCIFAR10, PermutedMNIST
from avalanche.benchmarks.generators import filelist_benchmark, dataset_benchmark, \
                                            tensors_benchmark, paths_benchmark

from avalanche.models import pytorchcv_wrapper, SimpleMLP, resnet32, SimpleCNN
from avalanche.training.determinism.rng_manager import RNGManager

from avalanche.training.supervised import Naive
from avalanche.training.plugins import ReplayPlugin, EvaluationPlugin

from avalanche.evaluation.metrics import (
    forgetting_metrics,
    accuracy_metrics,
    loss_metrics,
)
from avalanche.logging import InteractiveLogger

In [ ]:
# Remember to load checkpoints by setting the same random seed used when creating them...
RNGManager.set_random_seeds(1234)

In [ ]:
def split_dataset_in_task(dataframe_train, dataframe_test, class_category):

  train_class = dataframe_train.loc[dataframe_train['label']==class_category]
  test_class = dataframe_test.loc[dataframe_test['label']==class_category]

  return train_class, test_class

exp = {'Random': {'Dataframe': train_exp_rnd},
       'High':   {'Dataframe': train_exp_high},
       'Low':    {'Dataframe': train_exp_low}}

condition = "Random"

#a single train class is 800. A single test class is 200
train_class1, test_class1 = split_dataset_in_task(exp.get(condition)['Dataframe'], test_exp, 'animal')
train_class2, test_class2 = split_dataset_in_task(exp.get(condition)['Dataframe'], test_exp, 'food')
train_class3, test_class3 = split_dataset_in_task(exp.get(condition)['Dataframe'], test_exp, 'landscape')
train_class4, test_class4 = split_dataset_in_task(exp.get(condition)['Dataframe'], test_exp, 'sports')
train_class5, test_class5 = split_dataset_in_task(exp.get(condition)['Dataframe'], test_exp, 'vehicle')

In [ ]:
# --- CREATE CUSTOM DATASET
# The final dataset "train_experiences" is a list of 5 list (one for each class).
# Each list contains 2000 tuples (name of the file --> str, label --> int)

def create_tuples(dataframe):
    tuples_list = []
    for i, row in dataframe.iterrows():
        img_path = row["img_path"]
        cat = class_dict.get(row['label'])
        instance_tuple = (img_path, cat)
        tuples_list.append(instance_tuple)

    return tuples_list


In [ ]:
train_class1 = create_tuples(train_class1)
train_class2 = create_tuples(train_class2)
train_class3 = create_tuples(train_class3)
train_class4 = create_tuples(train_class4)
train_class5 = create_tuples(train_class5)

test_class1 = create_tuples(test_class1)
test_class2 = create_tuples(test_class2)
test_class3 = create_tuples(test_class3)
test_class4 = create_tuples(test_class4)
test_class5 = create_tuples(test_class5)

In [ ]:
train_task0 = torch.utils.data.DataLoader(train_class1 + train_class2 + train_class3)
train_task1 = torch.utils.data.DataLoader(train_class4 + train_class5)

test_task0 = torch.utils.data.DataLoader(test_class1 + test_class2 + test_class3)
test_task1 = torch.utils.data.DataLoader(test_class4 + test_class5)

### Create Avalanche Scenario

In [ ]:
# Data transformation
train_transform = Compose([
    Resize((224,224)),
    RandomHorizontalFlip(),
    RandomRotation(15),
    ToTensor(),
    Normalize((0.1307,), (0.3081,))  #[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
])

test_transform = Compose([
    Resize((224,224)),
    ToTensor(),
    Normalize((0.1307,), (0.3081,))  #[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
])

In [ ]:
#Here we create a GenericCLScenario object
scenario = paths_benchmark(
    [train_task0.dataset, train_task1.dataset],
    [test_task0.dataset, test_task1.dataset],
    task_labels=[0, 1],
    # complete_test_set_only=True,
    train_transform=train_transform,
    eval_transform=test_transform,
    )


train_stream = scenario.train_stream
test_stream = scenario.test_stream

#check if the scenario we designed is correct
for experience in train_stream:

    print("Start of task ", experience.task_label)
    print('Classes in this task:', experience.classes_in_this_experience)

    exp_id = experience.current_experience
    training_dataset = experience.dataset

    # The current Pytorch training set can be easily recovered through the experience
    current_training_set = experience.dataset
    #print("example of first training sample", experience.dataset[0])
    print('This task contains', len(current_training_set), 'training examples')

    # we can recover the corresponding test experience in the test stream
    current_test_set = test_stream[exp_id].dataset
    print('This task contains', len(current_test_set), 'test examples')

Start of task  0
Classes in this task: [0, 1, 2]
This task contains 2400 training examples
This task contains 600 test examples
Start of task  1
Classes in this task: [3, 4]
This task contains 1600 training examples
This task contains 400 test examples


In [ ]:
# --- CONFIG

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

results_path = "/content/drive/MyDrive/BMM_2023/projects/memorability_results/CL"
file_title = "ExpRandom__resnet18_pretrained_10"
file_name = os.path.join(results_path, file_title)

# Model getter: specify dataset and depth of the network.
#model = torch.hub.load('pytorch/vision:v0.10.0', 'squeezenet1_0', pretrained=False)
model = models.resnet18(pretrained=True)
optimizer = Adam(model.parameters(), lr=0.0005)
criterion = CrossEntropyLoss()

# choose some metrics and evaluation method
interactive_logger = InteractiveLogger()

eval_plugin = EvaluationPlugin(
    accuracy_metrics(minibatch=True, epoch=True, experience=True, stream=True),
    loss_metrics(minibatch=True, epoch=True, experience=True, stream=True),
    forgetting_metrics(experience=True),
    loggers=[interactive_logger],
)

#create strategy
strategy = Naive(model=model,
                optimizer=optimizer,
                criterion=criterion,
                train_mb_size=128,
                eval_mb_size=128,
                train_epochs=10,
                device=device,

                # plugins=[ReplayPlugin(mem_size=1000)],  #Naive, with Replay
                evaluator=eval_plugin)




with open(file_name + '.pkl', 'wb') as file:

    print("Starting experiment...")
    results = []

    #train on the selected benchmark with the chosen strategy
    #each "experience" here is a task
    for experience in train_stream:
        print("Start training on task ", experience.current_experience)
        strategy.train(experience, num_workers=10)
        print("Training completed")

        print("Computing accuracy on the whole test set")
        results.append(strategy.eval(test_stream, num_workers=10))

    df_results = pd.DataFrame(results)

    pickle.dump(df_results, file)
    print(f"Data saved to {file_name}")

Using device: cuda


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Starting experiment...
Start training on task  0
-- >> Start of training phase << --
0it [00:00, ?it/s]

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:560: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


100%|██████████| 19/19 [01:53<00:00,  5.99s/it]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 1.2657
	Loss_MB/train_phase/train_stream/Task000 = 0.2163
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.8483
	Top1_Acc_MB/train_phase/train_stream/Task000 = 0.9688
100%|██████████| 19/19 [00:27<00:00,  1.43s/it]
Epoch 1 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.1375
	Loss_MB/train_phase/train_stream/Task000 = 0.0614
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9637
	Top1_Acc_MB/train_phase/train_stream/Task000 = 0.9688
100%|██████████| 19/19 [00:24<00:00,  1.30s/it]
Epoch 2 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0801
	Loss_MB/train_phase/train_stream/Task000 = 0.0888
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.9754
	Top1_Acc_MB/train_phase/train_stream/Task000 = 0.9688
100%|██████████| 19/19 [00:27<00:00,  1.42s/it]
Epoch 3 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.0252
	Loss_MB/train_phase/train_stream/Task00

In [ ]:
results

[{'Top1_Acc_MB/train_phase/train_stream/Task000': 1.0,
  'Loss_MB/train_phase/train_stream/Task000': 0.00973567832261324,
  'Top1_Acc_Epoch/train_phase/train_stream/Task000': 0.9945833333333334,
  'Loss_Epoch/train_phase/train_stream/Task000': 0.01649930587038398,
  'Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000': 0.975,
  'Loss_Exp/eval_phase/test_stream/Task000/Exp000': 0.13156572217742601,
  'Top1_Acc_Exp/eval_phase/test_stream/Task001/Exp001': 0.0,
  'Loss_Exp/eval_phase/test_stream/Task001/Exp001': 15.71083854675293,
  'Top1_Acc_Stream/eval_phase/test_stream/Task001': 0.585,
  'Loss_Stream/eval_phase/test_stream/Task001': 6.363274852007628},
 {'Top1_Acc_MB/train_phase/train_stream/Task000': 1.0,
  'Loss_MB/train_phase/train_stream/Task000': 0.00973567832261324,
  'Top1_Acc_Epoch/train_phase/train_stream/Task000': 0.9945833333333334,
  'Loss_Epoch/train_phase/train_stream/Task000': 0.01649930587038398,
  'Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000': 0.0,
  'Loss_Exp/ev